# 第1回　ガイダンス：統計学Ⅰの復習と本講義の位置づけ
## ―― Ⅰ＝直感の敗北、Ⅱ＝集団の敗北

統計学Ⅱ　2026後期　／　北星学園大学　／　小野原 彩香

---

### このノートの使い方

**今日もプログラミングはしない。** 各セルの左の ▶ ボタンを上から順に押して、結果を **自分の目で見る** だけでよい。

- **統計学Ⅰを受けた人**：Ⅰでやったこと（代表値・ばらつき・相関）を高速で思い出す回。**Ⅰとまったく同じデータを使う**ので、見覚えのある数字が出てくるはず。
- **Ⅱから来た人**：Colab に慣れるための回。▶ を押すだけで大丈夫。

そして最後に、**Ⅱのクライマックス（第13回）の予告編** を一瞬だけ見る ―― 「賢い人を大勢集めれば正しく決められる。ただし *独立に* 判断すれば」。

---
## 0. 準備 ―― 霊長類376種のデータを読み込む

統計学Ⅰと同じ、**世界中の霊長類376種**の実データを使う。生物学者が何十年もかけて発表してきた測定値を1つの表にまとめたものである。▶ を押すだけ。

In [ ]:
# 準備：ライブラリと、霊長類376種のデータを読み込む。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats

def _build_from_source():
    """公開データ（PanTHERIA）から、この授業で使う形に組み立て直す。"""
    # 原典（PanTHERIA）から組み立て直す。まずリポジトリ同梱の複製、だめなら発行元から。
    # 発行元は User-Agent を見て弾くことがあるため、明示して取得する。
    import io, urllib.request
    SRCS = [
        "https://raw.githubusercontent.com/aonoa68/toukei-2/main/docs/data/PanTHERIA_1-0_WR05_Aug2008.txt.gz",
        "https://esapubs.org/archive/ecol/E090/184/PanTHERIA_1-0_WR05_Aug2008.txt",
    ]
    fam = {"Cercopithecidae":"オナガザル科","Cebidae":"オマキザル科","Pitheciidae":"サキ科",
           "Atelidae":"クモザル科","Cheirogaleidae":"コビトキツネザル科","Lemuridae":"キツネザル科",
           "Galagidae":"ガラゴ科","Hylobatidae":"テナガザル科","Indriidae":"インドリ科",
           "Lorisidae":"ロリス科","Lepilemuridae":"イタチキツネザル科","Aotidae":"ヨザル科",
           "Hominidae":"ヒト科","Tarsiidae":"メガネザル科","Daubentoniidae":"アイアイ科"}
    cols = {"MSW05_Binomial":"学名","MSW05_Genus":"属","5-1_AdultBodyMass_g":"体重g",
            "13-1_AdultHeadBodyLen_mm":"頭胴長mm","5-3_NeonateBodyMass_g":"新生児体重g",
            "10-2_SocialGrpSize":"集団サイズ","9-1_GestationLen_d":"妊娠期間日",
            "25-1_WeaningAge_d":"離乳日齢","3-1_AgeatFirstBirth_d":"初産日齢",
            "14-1_InterbirthInterval_d":"出産間隔日","15-1_LitterSize":"一腹産子数",
            "17-1_MaxLongevity_m":"最長寿命月","22-1_HomeRange_km2":"行動圏km2",
            "21-1_PopulationDensity_n/km2":"個体群密度","26-1_GR_Area_km2":"分布域km2",
            "6-2_TrophicLevel":"栄養段階","12-1_HabitatBreadth":"生息環境幅",
            "28-2_Temp_Mean_01degC":"平均気温01","28-1_Precip_Mean_mm":"月降水量mm"}
    src = None
    for _url in SRCS:
        try:
            _req = urllib.request.Request(_url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(_req, timeout=60) as _r:
                _raw = _r.read()
            _comp = "gzip" if _url.endswith(".gz") else None
            src = pd.read_csv(io.BytesIO(_raw), sep="\t", compression=_comp)
            break
        except Exception:
            continue
    if src is None:
        raise RuntimeError("原典データを取得できませんでした")
    p = src[src["MSW05_Order"] == "Primates"]
    out = p[list(cols)].rename(columns=cols)
    out.insert(1, "科", p["MSW05_Family"].map(fam))
    t = out.pop("平均気温01")
    out["平均気温C"] = np.where(t == -999, -999, (t / 10).round(1))
    return out.replace(-999, np.nan).sort_values("学名").reset_index(drop=True)

try:
    df = pd.read_csv("https://aonoa68.github.io/toukei-2/data/primates.csv")
except Exception:
    df = _build_from_source()

print("種数:", len(df), " 科数:", df["科"].nunique())
df.head()

---
## 1. Ⅰの復習①　代表値の嘘 ―― 「平均◯◯」に騙されるな

霊長類の **体重** を見てみよう。

**問い：「代表的な霊長類」の体重を語るとき、平均と中央値、どちらが実態に近い？**

予想を決めてから ▶。

In [ ]:
w = df["体重g"].dropna()
平均 = w.mean()
中央値 = w.median()

print(f"体重の 平均　 ： {平均:,.0f} g")
print(f"体重の 中央値： {中央値:,.0f} g")
print(f"種数         ： {len(w)} 種")

plt.figure(figsize=(7, 4))
plt.hist(w, bins=40, color="#4dabb6", edgecolor="white")
plt.axvline(平均, color="#e8503a", ls="-", lw=2, label=f"平均 {平均:,.0f}g")
plt.axvline(中央値, color="#333", ls="--", lw=2, label=f"中央値 {中央値:,.0f}g")
plt.xlabel("体重（g）")
plt.ylabel("種数")
plt.title("霊長類の体重の分布")
plt.legend()
plt.show()

**平均は中央値のほぼ2倍。** 少数の大型類人猿（ゴリラ149kg など）が平均を右へ引っ張るからだ。

分布が歪んでいるとき、「平均◯◯」は *ほとんどの対象より高い* 値になりうる。**どの代表値を選ぶかは、すでに一つの主張**だ ―― これがⅠの第2回だった。

> Ⅰではさらに、この分布が**二山**であること（小型のサルの山と、オナガザル科の山）まで見た。> 平均も中央値も「よくいる霊長類」を指していなかった。

**Ⅱでは、この「分布の形」そのものを数理的に扱う。**なぜ正規分布が現れるのか、なぜ現れないのか ―― 第4〜5回で確率分布として定式化する。

---
## 2. Ⅰの復習②　相関 ―― ただし「相関」は「因果」ではない

Ⅰの第5回でいちばん引っかかったであろう例を、もう一度出す。

**離乳日齢**（子が乳離れするまでの日数）と **行動圏**（ふだん動きまわる範囲）の関係。

予想：相関はあると思う？　あるとして、それは「子育てが長いから広く動く」という *因果* だと言い切れる？

In [ ]:
e = df.dropna(subset=["離乳日齢", "行動圏km2"])
x, y = np.log10(e["離乳日齢"]), np.log10(e["行動圏km2"])
r = x.corr(y)

print(f"離乳日齢と行動圏の相関係数 r = {r:.2f}   (n={len(e)}種)")

plt.figure(figsize=(7, 4.5))
plt.scatter(x, y, s=18, alpha=0.6, color="#4dabb6")
plt.xlabel("離乳日齢（対数目盛り）")
plt.ylabel("行動圏（対数目盛り）")
plt.xticks([1.5, 2, 2.5, 3, 3.5], ["30日", "100日", "300日", "1000日", "3000日"])
plt.yticks([-2, -1, 0, 1], ["0.01km²", "0.1km²", "1km²", "10km²"])
plt.title(f"離乳日齢 と 行動圏（r = {r:.2f}）")
plt.show()

はっきりした正の相関がある。だが **相関は因果ではない**。

Ⅰで確かめたとおり、犯人は **体の大きさ** だった。大きい種は子育てが長く、同時に広く動く。体重を統制すると、この相関は消える。

In [ ]:
# 体重の効果を引き算してから、もう一度相関を測る（Ⅰ第5回の復習）
s = df.dropna(subset=["離乳日齢", "行動圏km2", "体重g"]).copy()
for col in ["離乳日齢", "行動圏km2", "体重g"]:
    s[col] = np.log10(s[col])

X = np.column_stack([np.ones(len(s)), s["体重g"]])
残差 = lambda v: s[v] - X @ np.linalg.lstsq(X, s[v], rcond=None)[0]

print(f"（同じ {len(s)} 種で比較）")
print(f"単純な相関   r = {s['離乳日齢'].corr(s['行動圏km2']):.3f}")
print(f"体重を統制後 r = {np.corrcoef(残差('離乳日齢'), 残差('行動圏km2'))[0,1]:.3f}   ← 消えた")

「もっともらしい発見」が、第三の要因（**交絡**）で消えた。

> Ⅰの第5回でやったこの問いを、Ⅱでは **第10〜11回（因果推論・交絡）** で数理的に深掘りする。> 観察データから因果を主張することの難しさが、今期の大きなテーマの一つだ。
> **統制すれば済むのか。統制し忘れた変数があったらどうなるのか。**

---
## 3. Ⅱの予告編　―― 「賢い集団」は本当に賢いのか？

ここからが統計学Ⅱの世界だ。

次のような状況を考える：

- ある問題に「正解」がある（YES か NO か）。
- 一人ひとりは完璧ではないが、**コインより少しだけ賢い**（正解率 55%）。
- その人たちを集めて **多数決** で答えを決める。

**問い：人数を増やすと、多数決の正解率はどうなる？**　55%のまま？　それとも上がる？

予想を決めてから ▶。

In [ ]:
# 正解率55%の人を n 人集めて多数決したときの『多数決の正解率』を測る
# 各人は『独立に』自分で判断する、という前提でシミュレーションする
r2 = np.random.default_rng(42)
p個人 = 0.55          # 一人の正解率
試行 = 20000           # 何回投票をやり直すか
人数リスト = [1, 3, 5, 11, 21, 51, 101, 201]

多数決正解率 = []
for n in 人数リスト:
    投票 = r2.random((試行, n)) < p個人      # True=その人は正解した（独立に判断）
    多数派が正解 = 投票.sum(axis=1) > n / 2   # 過半数が正解なら多数決も正解
    多数決正解率.append(多数派が正解.mean())

for n, acc in zip(人数リスト, 多数決正解率):
    print(f"{n:>4} 人で多数決 → 正解率 {acc:.1%}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(人数リスト, 多数決正解率, marker="o", color="#e8503a")
plt.axhline(0.55, ls="--", color="gray", label="一人の正解率 55%")
plt.ylim(0.4, 1.0)
plt.xlabel("投票する人数")
plt.ylabel("多数決の正解率")
plt.title("独立に判断する人を増やすと、多数決はほぼ確実に正解する")
plt.legend()
plt.show()

たった55%の人たちでも、**独立に**判断して数を増やせば、多数決の正解率は **ほぼ100%** に近づく。これが **コンドルセの陪審定理**（1785年）―― 「群衆の智慧」の数学的な正体だ。

> ⚠️ **ただし、ここに罠がある**
> 
> いま一行だけ、こっそり強い仮定を置いた ―― **「各人が *独立に* 判断する」**。
> 
> もし人々が互いに顔色をうかがい、**「空気を読んで」** 多数派に合わせ始めたら？　この定理は **崩壊する**。賢いはずの集団が、全員そろって間違える。
> 
> なぜそうなるのか、どれくらい正解率が落ちるのかを、**第13回「『空気を読む』ことの愚かさ：コンドルセの陪審定理」** でシミュレーションして確かめる。今日はその予告編だ。

### 独立性は、今日すでに2回出てきている

気づいただろうか。**今日の3つの話は、すべて「独立」でつながっている。**

| 場面 | 独立性はどこに関わるか |
|---|---|
| §1 代表値 | 平均が信用できるのは、標本が独立に選ばれているとき（Ⅰ第6回） |
| §2 交絡 | 離乳日齢と行動圏は**独立ではなかった**。両方を体重が動かしていた |
| §3 多数決 | 各人の判断が独立なら集合知になる。独立でなければ崩壊する |

**今期を貫く一本の糸は「独立性」である。**

---
## 今日のまとめ

| | Ⅰで学んだこと | Ⅱで深掘りすること |
|---|---|---|
| 代表値 | 平均は外れ値に弱い・分布の形を見る | 確率分布の数理（第4〜5回） |
| 相関 | 相関 ≠ 因果（体サイズによる交絡） | 因果推論・交絡（第10〜11回） |
| 標本 | CLTで一部から全体を語れる | 独立が壊れたときのCLT（第6〜7回） |
| 集団 | （Ⅰでは扱わない） | **独立性と集合知（第12〜13回）** |

Ⅰは「**あなたの直感**は系統的に外れる」を示した。
Ⅱは「**集団になると、もっと外れる**（空気を読むと）」を、確率論で証明していく。

今期を貫く一本の糸は **「独立性」** だ。

> **課題（Moodle）**：Ⅰの要点クイズ（自動採点）＋「Ⅰで一番『直感が外れた』のはどこか／Ⅱに何を期待するか」のふりかえり（記述）。Ⅱから来た人は『統計に対する今の自分の構え』を書く。詳しくはMoodleの第1回課題を見ること。

---

!!! quote "このデータの出典"
    Jones, K.E. et al. (2009) PanTHERIA: a species-level database of life history,
    ecology, and geography of extant and recently extinct mammals.
    *Ecology* 90(9): 2648. Ecological Archives E090-184.

    霊長類376種の行だけを抜き出し、列を選び、気温の単位を直したもの。値は変えていない。